# 08 — Mô hình Transformer

Tìm cấu hình tốt nhất trên **Validation** (09/2024–08/2025), rồi huấn luyện lại với 3 seed.
Mô hình báo cáo là trung bình dự báo của 3 seed. **Test không được dùng ở notebook này**, chỉ dùng ở 09.

GRU và Transformer dùng chung: bộ feature của 06, cách chia dữ liệu, chuẩn hóa, tập mẫu đánh giá,
số lần thử (`N_TRIALS`), early stopping và hàm loss (MAE). Chỉ kiến trúc mạng là khác.

In [ ]:
# Chạy trên Colab (GPU): bỏ comment, sửa link repo của bạn
# !git clone <link-repo> repo
# %cd repo/notebooks

## I. Thư viện và dữ liệu

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import forecasting as fc

fc.enable_determinism()
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
feature_config = fc.load_feature_config(ROOT)
FEATURES = feature_config['selected_features']
WINDOW_06 = int(feature_config['input_window'])

data = fc.WindowData(ROOT, FEATURES)
print('Feature (từ 06):', len(FEATURES), FEATURES)
print('Window 06 chọn:', WINDOW_06)
print('Ngưỡng PM cao (P90 của Train):', round(data.high_threshold, 1))
display(data.summary())

## II. Không gian tìm kiếm

- `window`: số giờ lịch sử đưa vào mô hình.
- `target_mode`: `level` dự báo thẳng PM2.5; `delta` dự báo mức thay đổi so với PM2.5 hiện tại (thường tốt hơn ở horizon ngắn).
- Random search `N_TRIALS` cấu hình, mỗi cấu hình chạy 1 lần (seed 42). Kết quả được lưu sau mỗi lần thử,
  nên nếu Colab bị ngắt thì chạy lại cell sẽ tiếp tục từ chỗ dừng.

In [ ]:
N_TRIALS = 20
FIT_KW = dict(max_epochs=100, patience=10, batch=256)
FINAL_SEEDS = [42, 123, 2024]
OUT_DIR = ROOT / 'results' / 'models' / 'transformer'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPACE = {
    'window': sorted({24, 48, 72, 96, 168, WINDOW_06}),
    'target_mode': ['level', 'delta'],
    'd_model': [32, 64],
    'heads': [2, 4],
    'layers': [1, 2, 3],
    'ff_mult': [2, 4],
    'dropout': [0.1, 0.2],
    'dense': [32, 64],
    'lr': [1e-3, 5e-4],
}

# Luôn thử cấu hình gần với GRU nhỏ đã dùng ở 06 để làm mốc
ALWAYS = [{'window': WINDOW_06, 'target_mode': 'level', 'd_model': 32, 'heads': 2, 'layers': 1,
           'ff_mult': 2, 'dropout': 0.1, 'dense': 32, 'lr': 1e-3}]

configs = fc.sample_configs(SPACE, N_TRIALS, seed=0, always=ALWAYS)
print('Số cấu hình sẽ thử:', len(configs))

## III. Tìm cấu hình trên Validation

In [ ]:
tuning = fc.tune('transformer', configs, data, OUT_DIR / 'tuning.csv', seed=42, **FIT_KW)
show = [c for c in tuning.columns if c not in ('config', 'trial')]
display(tuning[show].head(10).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for mode, g in tuning.groupby('target_mode'):
    axes[0].scatter(g['window'], g['val_MAE'], label=mode)
axes[0].set_xlabel('Window (giờ)'); axes[0].set_ylabel('Val MAE (µg/m³)'); axes[0].legend(title='target')
axes[1].scatter(tuning['n_params'], tuning['val_MAE'])
axes[1].set_xscale('log'); axes[1].set_xlabel('Số tham số'); axes[1].set_ylabel('Val MAE (µg/m³)')
plt.tight_layout(); plt.show()

## IV. Huấn luyện cấu hình tốt nhất với 3 seed

In [ ]:
best_cfg = json.loads(tuning.iloc[0]['config'])
print('Cấu hình tốt nhất:', best_cfg)

ensemble, seed_table = fc.train_final('transformer', best_cfg, data, FINAL_SEEDS, OUT_DIR, 'Transformer', **FIT_KW)
display(seed_table[seed_table['Split'] == 'Validation'].round(3))

## V. Kết quả trên Validation so với mốc đơn giản

In [ ]:
pos = data.positions['Validation']
y_val = data.targets(pos)
rows = []
for method, pred in {'Transformer': ensemble['Validation'], **fc.baselines(data, 'Validation')}.items():
    t = fc.metrics_table(y_val, pred, data)
    t.insert(0, 'Method', method)
    rows.append(t)
val_table = pd.concat(rows, ignore_index=True)
val_table.to_csv(OUT_DIR / 'validation_metrics.csv', index=False)
display(val_table.pivot(index='Method', columns='Horizon', values='MAE').round(2))